## seed_dim_date
Populates `silver.dim_date` at month-end grain (quarter-ends are the Mar/Jun/Sep/Dec month-ends, so this single grain serves both the monthly facts and the quarterly FHFA fact). Range starts 1947 (CPI history) and runs to 2031. Idempotent MERGE on `date_key` (deterministic yyyymmdd), so geo/date keys stay stable across re-seeds.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects SILVER, F, spark. date_key = yyyymmdd of the (month-end) full_date.
START = "1947-01-01"   # CPI series reaches back to 1947; covers every source
END   = "2031-12-01"

cal = spark.sql(
    f"SELECT explode(sequence(to_date('{START}'), to_date('{END}'), interval 1 month)) AS m"
)
dim = (
    cal.select(F.last_day("m").alias("full_date"))
    .select(
        (F.year("full_date") * 10000 + F.month("full_date") * 100
         + F.dayofmonth("full_date")).cast("int").alias("date_key"),
        F.col("full_date"),
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.trunc("full_date", "MM").alias("month_start"),
        F.trunc("full_date", "quarter").alias("quarter_start"),
        F.lit(True).alias("is_month_end"),
        F.month("full_date").isin(3, 6, 9, 12).alias("is_quarter_end"),
    )
)
dim.createOrReplaceTempView("dim_date_staging")

spark.sql(f"""
    MERGE INTO {SILVER}.dim_date t USING dim_date_staging s ON t.date_key = s.date_key
    WHEN MATCHED THEN UPDATE SET
        t.full_date=s.full_date, t.year=s.year, t.quarter=s.quarter, t.month=s.month,
        t.month_start=s.month_start, t.quarter_start=s.quarter_start,
        t.is_month_end=s.is_month_end, t.is_quarter_end=s.is_quarter_end,
        t.updated_ts=current_timestamp()
    WHEN NOT MATCHED THEN INSERT
        (date_key, full_date, year, quarter, month, month_start, quarter_start,
         is_month_end, is_quarter_end, inserted_ts, updated_ts)
        VALUES (s.date_key, s.full_date, s.year, s.quarter, s.month, s.month_start,
                s.quarter_start, s.is_month_end, s.is_quarter_end,
                current_timestamp(), current_timestamp())
""")
row_count = spark.table(f"{SILVER}.dim_date").count()
print(f"seed_dim_date: dim_date rows = {row_count:,}")